# Feature Attribution: Explaining Individual Income Predictions
## Breaking down each prediction into per-feature contributions

When a model predicts that a customer earns $82,000, which features are
responsible — and by how much? This notebook implements three complementary
explanation methods from scratch:

| # | Method | Scope | Key idea |
|---|---|---|---|
| 1 | Linear attribution (exact) | Per prediction | Ridge coefficients × feature values |
| 2 | Permutation importance | Global + per store | Shuffle one feature, measure MAE increase |
| 3 | LIME-style local explanation | Per prediction | Fit a local linear model around each point |

Together they answer:
- **What features matter most globally?** (permutation importance)
- **What features drove *this specific* prediction?** (linear attribution)
- **Why is this prediction wrong?** (attribution on high-error customers)
- **Are bad predictions caused by the same features?** (error correlation analysis)

## 1. Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from scipy.stats import rankdata, spearmanr, pearsonr
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

STORES  = ["whole_foods","kroger","safeway","walmart","thrift_store"]
LABELS  = ["Whole Foods","Kroger","Safeway","Walmart","Thrift"]
TIERS   = ["High","Median","Median","Median","Low"]
COLORS  = ["#185FA5","#1D9E75","#0F6E56","#854F0B","#A32D2D"]
FEAT_COLS = ["age","visits_per_month","avg_basket_usd","monthly_spend_usd",
             "grocery_pct","electronics_pct","apparel_pct","home_pct",
             "private_label_pct","online_orders_pct","coupon_usage_pct",
             "loyalty_score","segment_enc"]
FEAT_NICE = ["Age","Visits/mo","Basket $","Monthly spend","Grocery %",
             "Elec %","Apparel %","Home %","Private label","Online %",
             "Coupon %","Loyalty","Segment"]
PROXY_IDX = [2, 9]
BEST_ALPHAS = [0.1, 0.3, 0.5, 0.7]

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5, "font.size": 10,
})
print("Setup complete.")

## 2. Load data and run the S10 rank model pipeline

In [ ]:
df = pd.read_csv("grocery_all_stores.csv")
le = LabelEncoder(); le.fit(df["segment"]); df["segment_enc"] = le.transform(df["segment"])

TRUE_RANGES = {"whole_foods":(85000,160000),"kroger":(55000,105000),
               "safeway":(45000,95000),"walmart":(30000,80000),"thrift_store":(10000,45000)}

sc_all    = StandardScaler()
X_all_sc  = sc_all.fit_transform(df[FEAT_COLS].values.astype(float))
pca_ref   = PCA(n_components=6, random_state=42)
X_all_pca = pca_ref.fit_transform(X_all_sc)
ref_lib   = {s: {"centroid": X_all_pca[df["store"]==s].mean(axis=0),
                 "income_mean": np.mean(TRUE_RANGES[s])} for s in STORES}

def make_bridge(Xs, Xt, yr, alphas=BEST_ALPHAS, n=400, seed=42):
    rng = np.random.RandomState(seed); bX, by = [], []
    for a in alphas:
        for _ in range(n//len(alphas)):
            i,j = rng.randint(len(Xs)), rng.randint(len(Xt))
            bX.append((1-a)*Xs[i]+a*Xt[j]); by.append((1-a)*yr[i]+a*0.5)
    return np.array(bX), np.array(by)

def get_range(Xs, Xt, ys, ts, refs):
    preg = LinearRegression().fit(Xs[:,PROXY_IDX], ys)
    gap  = preg.predict(Xt[:,PROXY_IDX]).mean() - preg.predict(Xs[:,PROXY_IDX]).mean()
    cen  = X_all_pca[df["store"]==ts].mean(axis=0).reshape(1,-1)
    dists = {s: cdist(cen, ref_lib[s]["centroid"].reshape(1,-1),"euclidean")[0,0] for s in refs}
    nn    = min(dists, key=dists.get)
    bl    = 0.4*ref_lib[nn]["income_mean"] + 0.6*(preg.predict(Xs[:,PROXY_IDX]).mean()+gap)
    span  = (ys.max()-ys.min())/2*1.1
    return max(0,round((bl-span)/1000)*1000), round((bl+span)/1000)*1000

def train_rank_model(Xs, Xt, ys, lo, hi):
    yr = (rankdata(ys)-1)/(len(ys)-1)
    bX, by = make_bridge(Xs, Xt, yr)
    m = Ridge(alpha=10.0)
    m.fit(np.vstack([Xs,bX]), np.concatenate([yr,by]))
    preds = lo + np.clip(m.predict(Xt),0,1)*(hi-lo)
    return preds, m, yr

# Run LOSO pipeline — collect everything needed for attribution
pipeline = {}
for ts in STORES:
    src  = df[df["store"]!=ts].copy()
    tgt  = df[df["store"]==ts].copy()
    Xsr  = src[FEAT_COLS].values.astype(float)
    Xtr  = tgt[FEAT_COLS].values.astype(float)
    ys   = src["income_usd"].values
    yt   = tgt["income_usd"].values
    sc   = StandardScaler(); Xs = sc.fit_transform(Xsr); Xt = sc.transform(Xtr)
    refs = [s for s in STORES if s!=ts]
    lo, hi = get_range(Xs, Xt, ys, ts, refs)
    preds, model, yr_src = train_rank_model(Xs, Xt, ys, lo, hi)
    errors = np.abs(yt - preds)
    pipeline[ts] = {"Xs":Xs,"Xt":Xt,"ys":ys,"yt":yt,"sc":sc,"model":model,
                    "yr_src":yr_src,"lo":lo,"hi":hi,"preds":preds,"errors":errors,
                    "tgt_df":tgt.reset_index(drop=True)}
    mae = errors.mean()
    print(f"  {LABELS[STORES.index(ts)]:<15}  MAE=${mae:>8,.0f}  range=${lo:,}-${hi:,}")

## 3. Method 1 — Linear attribution (exact, per prediction)

Because we use **Ridge regression**, the prediction is a linear function of the
standardised features:

```
rank_pred = intercept + w_1*x_1 + w_2*x_2 + ... + w_13*x_13
income_pred = lo + rank_pred * (hi - lo)
```

The contribution of feature j to prediction i is simply:

```
contribution_j(i) = w_j * x_j(i) * (hi - lo)
```

This decomposes the income prediction into exactly 13 additive pieces plus a baseline.
The pieces sum exactly to the prediction (no approximation).

**Interpretation:**
- Positive contribution: this feature pushed the prediction *higher*
- Negative contribution: this feature pushed the prediction *lower*
- Large absolute contribution: this feature has strong influence on this prediction

In [ ]:
def compute_attributions(model, Xt, lo, hi):
    """
    Decompose each prediction into per-feature contributions.
    Returns matrix of shape (n_customers, n_features).
    Each row sums to (pred_income - baseline_income).
    """
    w    = model.coef_           # shape (n_features,)
    b    = model.intercept_      # scalar
    span = hi - lo

    # Per-feature rank contribution (in rank units)
    rank_contribs = Xt * w[np.newaxis, :]    # (n, n_features)

    # Convert rank contributions to income contributions
    income_contribs = rank_contribs * span   # (n, n_features)

    # Baseline: what the model predicts for a "zero" standardised customer
    # (all features at source mean) — this is the intercept contribution
    baseline_income = lo + np.clip(b, 0, 1) * span

    return income_contribs, baseline_income

# Compute for all stores
for ts in STORES:
    r = pipeline[ts]
    contribs, baseline = compute_attributions(r["model"], r["Xt"], r["lo"], r["hi"])
    pipeline[ts]["contribs"]  = contribs       # (250, 13)
    pipeline[ts]["baseline"]  = baseline
    # Verify: baseline + sum(contribs) should equal preds
    reconstructed = baseline + contribs.sum(axis=1)
    max_err = np.abs(reconstructed - r["preds"]).max()
    print(f"  {LABELS[STORES.index(ts)]:<15}  attribution max recon error=${max_err:.2f}  (should be ~0)")

## 4. Global attribution — mean absolute contribution per feature

Averaging |contribution| across all customers gives the average feature importance
in absolute income terms ($). This is the attribution analogue of feature importance.

In [ ]:
# ── Plot 1: mean |contribution| heatmap across stores ─────────────────────────
mean_abs = np.array([np.abs(pipeline[ts]["contribs"]).mean(axis=0) for ts in STORES])
# shape (5 stores, 13 features)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Mean absolute feature contribution ($ impact on income prediction)",
             fontweight="bold")

# Heatmap
ax = axes[0]
im = ax.imshow(mean_abs / 1000, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(FEAT_COLS))); ax.set_xticklabels(FEAT_NICE, rotation=60, ha="right", fontsize=9)
ax.set_yticks(range(len(STORES)));   ax.set_yticklabels(LABELS, fontsize=10)
for j, col in enumerate(COLORS): ax.get_yticklabels()[j].set_color(col)
for r in range(len(STORES)):
    for c in range(len(FEAT_COLS)):
        v = mean_abs[r,c]/1000
        ax.text(c, r, f"{v:.1f}", ha="center", va="center",
                fontsize=7, color="white" if v > mean_abs.max()/1000*0.65 else "black")
plt.colorbar(im, ax=ax, label="Mean |contribution| ($k)", shrink=0.8)
ax.set_title("Mean |attribution| heatmap ($k)", fontweight="bold")

# Grouped bar — top features
ax = axes[1]
top_feats = np.argsort(mean_abs.mean(axis=0))[::-1][:8]
x = np.arange(len(top_feats)); w = 0.14
for i, (ts, lbl, col) in enumerate(zip(STORES, LABELS, COLORS)):
    ax.bar(x + (i-2)*w, mean_abs[i, top_feats]/1000, w,
           label=lbl, color=col, edgecolor="white", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels([FEAT_NICE[j] for j in top_feats], rotation=30, ha="right")
ax.set_ylabel("Mean |contribution| ($k)")
ax.set_title("Top 8 features by mean absolute attribution", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Waterfall chart — explaining one single prediction

A waterfall chart shows exactly how each feature moved the prediction
from the baseline (source-average customer) to the final predicted income.
Green bars push the prediction up; red bars push it down.

In [ ]:
def waterfall(ax, contribs, feat_names, baseline, pred, actual,
              title="", color="steelblue"):
    """Draw a waterfall chart for one prediction."""
    # Sort by absolute contribution, keep top 10
    order = np.argsort(np.abs(contribs))[::-1][:10]
    vals  = contribs[order]
    names = [feat_names[i] for i in order]

    cumsum  = baseline
    xpos    = 0
    bottoms = []
    heights = []
    colors  = []

    for v in vals:
        bottoms.append(cumsum if v > 0 else cumsum + v)
        heights.append(abs(v))
        colors.append("#27500A" if v > 0 else "#791F1F")
        cumsum += v

    bars = ax.bar(range(len(vals)), [h/1000 for h in heights],
                  bottom=[b/1000 for b in bottoms],
                  color=colors, edgecolor="white", linewidth=0.5, width=0.6)

    ax.axhline(baseline/1000, color="gray", linewidth=1, linestyle=":",
               label=f"Baseline ${baseline/1000:.0f}k")
    ax.axhline(pred/1000,     color=color,  linewidth=1.5, linestyle="--",
               label=f"Pred ${pred/1000:.0f}k")
    ax.axhline(actual/1000,   color="black", linewidth=1.5,
               label=f"Actual ${actual/1000:.0f}k")

    # Value labels on bars
    for bar, v in zip(bars, vals):
        y_mid = bar.get_y() + bar.get_height()/2
        ax.text(bar.get_x()+bar.get_width()/2, y_mid,
                f"${v/1000:+.1f}k", ha="center", va="center",
                fontsize=7.5, color="white", fontweight="bold")

    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=40, ha="right", fontsize=8.5)
    ax.set_ylabel("Income ($k)")
    ax.set_title(title, fontweight="bold", fontsize=9, color=color)
    ax.legend(fontsize=7.5, loc="upper right")

# Pick one good and one bad prediction for each of two stores
fig = plt.figure(figsize=(16, 10))
fig.suptitle("Waterfall charts — single prediction breakdowns", fontweight="bold")

plot_cases = [
    ("whole_foods", "Whole Foods — low-error example",  "low",  COLORS[0]),
    ("whole_foods", "Whole Foods — high-error example", "high", COLORS[0]),
    ("thrift_store","Thrift Store — low-error example",  "low", COLORS[4]),
    ("thrift_store","Thrift Store — high-error example", "high", COLORS[4]),
]

for idx, (ts, title, err_type, col) in enumerate(plot_cases):
    r     = pipeline[ts]
    errs  = r["errors"]
    if err_type == "low":
        cust_i = int(np.argsort(errs)[5])     # 5th lowest error
    else:
        cust_i = int(np.argsort(errs)[-5])    # 5th highest error

    ax = fig.add_subplot(2, 2, idx+1)
    waterfall(ax,
              contribs  = r["contribs"][cust_i],
              feat_names= FEAT_NICE,
              baseline  = r["baseline"],
              pred      = r["preds"][cust_i],
              actual    = r["yt"][cust_i],
              title     = f"{title}\nerror=${errs[cust_i]/1000:.1f}k",
              color     = col)

plt.tight_layout()
plt.show()

## 6. Why are high-error predictions wrong? — attribution on bad predictions

Compare the feature attributions for the worst 20% predictions vs the best 20%.
If certain features appear consistently larger (or in the wrong direction)
for bad predictions, those features are the root cause of poor performance.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle("Attribution comparison: top-20% errors vs bottom-20% errors",
             fontweight="bold")

for col_i, (ts, lbl, col) in enumerate(zip(STORES, LABELS, COLORS)):
    r       = pipeline[ts]
    errs    = r["errors"]
    contribs= r["contribs"]
    n_20pct = max(1, len(errs)//5)

    low_idx  = np.argsort(errs)[:n_20pct]    # best predictions
    high_idx = np.argsort(errs)[-n_20pct:]   # worst predictions

    mean_low  = contribs[low_idx].mean(axis=0)
    mean_high = contribs[high_idx].mean(axis=0)
    diff      = mean_high - mean_low          # what extra attribution high-error has

    # Row 0: mean contribution (low vs high error)
    ax = axes[0, col_i]
    x  = np.arange(len(FEAT_COLS))
    ax.bar(x-0.2, mean_low/1000,  0.35, label="Low error",  color=col, alpha=0.5, edgecolor="white")
    ax.bar(x+0.2, mean_high/1000, 0.35, label="High error", color=col, alpha=0.95, edgecolor="white")
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_xticks(x); ax.set_xticklabels(FEAT_NICE, rotation=70, ha="right", fontsize=7)
    ax.set_title(f"{lbl}", fontweight="bold", color=col, fontsize=9)
    if col_i == 0: ax.set_ylabel("Mean contribution ($k)")
    ax.legend(fontsize=7)

    # Row 1: difference (high_error_contrib - low_error_contrib)
    ax = axes[1, col_i]
    diff_cols = ["#27500A" if d > 0 else "#791F1F" for d in diff]
    ax.bar(x, diff/1000, color=diff_cols, edgecolor="white", alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.7)
    # Label top-3 by absolute difference
    top3 = np.argsort(np.abs(diff))[-3:]
    for ti in top3:
        ax.text(ti, diff[ti]/1000 + (0.3 if diff[ti]>0 else -0.4),
                FEAT_NICE[ti], ha="center", fontsize=7, color="black", fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(FEAT_NICE, rotation=70, ha="right", fontsize=7)
    if col_i == 0: ax.set_ylabel("Difference ($k)")
    ax.set_title("High-error minus low-error attribution", fontsize=8, color=col)

plt.tight_layout()
plt.show()

print("\nTop features causing high errors (mean attribution difference):")
for ts, lbl, col in zip(STORES, LABELS, COLORS):
    r = pipeline[ts]; errs = r["errors"]; contribs = r["contribs"]
    n = max(1, len(errs)//5)
    diff = contribs[np.argsort(errs)[-n:]].mean(0) - contribs[np.argsort(errs)[:n]].mean(0)
    top = np.argsort(np.abs(diff))[-3:][::-1]
    print(f"  {lbl:<15}  " + "  |  ".join(f"{FEAT_NICE[i]}={diff[i]/1000:+.1f}k" for i in top))

## 7. Error-feature correlation — which feature attributions correlate with prediction error?

For each feature, compute the Spearman correlation between its attribution
(contribution to predicted income) and the actual prediction error.

A high positive correlation means: when this feature pushes the prediction *up*,
the model is more likely to be wrong (over-predicting).
A high negative correlation means: when this feature pushes the prediction *down*,
the model under-predicts.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4.5))
fig.suptitle("Spearman correlation: feature attribution vs prediction error",
             fontweight="bold")

for ax, ts, lbl, col in zip(axes, STORES, LABELS, COLORS):
    r        = pipeline[ts]
    contribs = r["contribs"]    # (250, 13)
    errors   = r["errors"]      # (250,) — absolute errors

    # Signed error: positive = over-predict, negative = under-predict
    signed_errors = r["preds"] - r["yt"]

    corrs = [float(spearmanr(contribs[:,j], signed_errors)[0])
             for j in range(len(FEAT_COLS))]

    sorted_idx = np.argsort(corrs)
    bar_cols   = ["#27500A" if c > 0 else "#791F1F" for c in [corrs[i] for i in sorted_idx]]
    bars = ax.barh([FEAT_NICE[i] for i in sorted_idx],
                   [corrs[i]     for i in sorted_idx],
                   color=bar_cols, edgecolor="white", height=0.65)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.axvline( 0.2, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.axvline(-0.2, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_title(f"{lbl}", fontweight="bold", color=col, fontsize=9)
    ax.set_xlabel("Spearman r with signed error")
    for bar, v in zip(bars, [corrs[i] for i in sorted_idx]):
        if abs(v) > 0.15:
            ax.text(v + (0.01 if v>0 else -0.01), bar.get_y()+bar.get_height()/2,
                    f"{v:.2f}", va="center", ha="left" if v>0 else "right", fontsize=7.5)
    ax.set_xlim(-0.6, 0.6)

plt.tight_layout(); plt.show()

print("Interpretation:")
print("  Positive r: when feature pushes prediction UP, model tends to OVER-predict.")
print("  Negative r: when feature pushes prediction DOWN, model tends to UNDER-predict.")
print("  Near zero:  this feature attribution is not systematically linked to errors.")

## 8. Method 2 — Permutation importance (global, model-agnostic)

Permutation importance does not use model weights.
Instead, it shuffles one feature at a time and measures how much the MAE increases.

```
importance(feature j) = MAE_with_j_shuffled - MAE_baseline
```

Large increase = feature j was important.
Near zero = feature j was not contributing (or was redundant with other features).

This method works with any model (Ridge, GBR, XGBoost, etc.) and does not
require access to model weights — only predictions.

In [ ]:
def permutation_importance(model, Xt, yt, lo, hi, n_repeats=10, seed=42):
    """
    Compute permutation importance for each feature.
    Returns mean and std of MAE increase across n_repeats shuffles.
    """
    rng      = np.random.RandomState(seed)
    base_mae = mean_absolute_error(yt, lo + np.clip(model.predict(Xt),0,1)*(hi-lo))
    n_feats  = Xt.shape[1]
    imp_mean = np.zeros(n_feats)
    imp_std  = np.zeros(n_feats)

    for j in range(n_feats):
        deltas = []
        for _ in range(n_repeats):
            Xt_perm = Xt.copy()
            Xt_perm[:, j] = rng.permutation(Xt_perm[:, j])
            perm_mae = mean_absolute_error(
                yt, lo + np.clip(model.predict(Xt_perm),0,1)*(hi-lo))
            deltas.append(perm_mae - base_mae)
        imp_mean[j] = np.mean(deltas)
        imp_std[j]  = np.std(deltas)

    return imp_mean, imp_std

# Compute for all stores
print("Computing permutation importance (may take ~30 seconds)...")
for ts in STORES:
    r = pipeline[ts]
    imp_mean, imp_std = permutation_importance(
        r["model"], r["Xt"], r["yt"], r["lo"], r["hi"], n_repeats=15)
    pipeline[ts]["perm_imp_mean"] = imp_mean
    pipeline[ts]["perm_imp_std"]  = imp_std
    top3 = np.argsort(imp_mean)[-3:][::-1]
    print(f"  {LABELS[STORES.index(ts)]:<15}  top: "
          + ", ".join(f"{FEAT_NICE[i]}(+${imp_mean[i]/1000:.1f}k)" for i in top3))

## 9. Permutation importance plots — per store and combined

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
fig.suptitle("Permutation importance: MAE increase when feature is shuffled ($k)",
             fontweight="bold")

for ax, ts, lbl, col in zip(axes, STORES, LABELS, COLORS):
    r    = pipeline[ts]
    imp  = r["perm_imp_mean"]
    std  = r["perm_imp_std"]
    si   = np.argsort(imp)[::-1]

    bar_cols = [col if v > 0 else "#B4B2A9" for v in imp[si]]
    bars = ax.barh([FEAT_NICE[i] for i in si], imp[si]/1000,
                   xerr=std[si]/1000, color=bar_cols[::-1][::-1],
                   edgecolor="white", height=0.65,
                   error_kw=dict(elinewidth=1, capsize=3, ecolor="gray"))
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("MAE increase ($k)")
    ax.set_title(f"{lbl}", fontweight="bold", color=col, fontsize=9)
    if ax == axes[0]: ax.set_ylabel("Feature")

plt.tight_layout(); plt.show()

# ── Combined: heatmap of permutation importance across all stores ─────────────
perm_mat = np.array([pipeline[ts]["perm_imp_mean"] for ts in STORES])  # (5, 13)

# Sort features by mean importance across stores
feat_order = np.argsort(perm_mat.mean(axis=0))[::-1]

fig, ax = plt.subplots(figsize=(13, 4))
im = ax.imshow(perm_mat[:, feat_order]/1000, cmap="YlOrRd", aspect="auto", vmin=0)
ax.set_xticks(range(len(FEAT_COLS)))
ax.set_xticklabels([FEAT_NICE[i] for i in feat_order], rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(STORES))); ax.set_yticklabels(LABELS, fontsize=10)
for j, col in enumerate(COLORS): ax.get_yticklabels()[j].set_color(col)
for r in range(len(STORES)):
    for c in range(len(FEAT_COLS)):
        v = perm_mat[r, feat_order[c]]/1000
        ax.text(c, r, f"{v:.1f}", ha="center", va="center", fontsize=8,
                color="white" if v > perm_mat.max()/1000*0.6 else "black")
plt.colorbar(im, ax=ax, label="MAE increase when shuffled ($k)", shrink=0.8)
ax.set_title("Permutation importance heatmap — sorted by mean importance across stores",
             fontweight="bold")
plt.tight_layout(); plt.show()

## 10. Method 3 — LIME-style local explanation

Linear attribution (Method 1) uses global model weights.
LIME fits a *local* linear model in the neighborhood of each prediction,
which can reveal interactions that the global weights miss.

**Algorithm for each customer point x:**
1. Generate n_samples perturbed versions of x by adding Gaussian noise
2. Get the rank model's predictions for all perturbed points
3. Weight each perturbed sample by its similarity to x (Gaussian kernel)
4. Fit a weighted Ridge regression: perturbed_features -> perturbed_rank_predictions
5. The coefficients of this local model are the LIME feature importances

Because the local model is fit only around x, it captures how the model
behaves *locally* — which may differ from the global behavior for
customers that are at the edges of the training distribution.

In [ ]:
def lime_explain(model, x, lo, hi, n_samples=300, kernel_width=0.75, seed=42):
    """
    LIME-style local explanation for a single prediction.

    Parameters
    ----------
    model        : trained rank model (Ridge)
    x            : single standardised feature vector (n_features,)
    lo, hi       : income range for rescaling
    n_samples    : number of perturbed samples to generate
    kernel_width : Gaussian kernel width (controls locality)

    Returns
    -------
    local_coefs  : local feature importances (n_features,)
    local_pred   : prediction at x from local model
    global_pred  : prediction at x from global model
    """
    rng = np.random.RandomState(seed)

    # Step 1: generate perturbed samples
    noise = rng.randn(n_samples, len(x)) * kernel_width
    X_pert = x[np.newaxis,:] + noise

    # Step 2: global model predictions on perturbed samples
    y_pert = lo + np.clip(model.predict(X_pert),0,1)*(hi-lo)

    # Step 3: similarity weights (Gaussian kernel in feature space)
    dists   = np.linalg.norm(noise, axis=1)
    weights = np.exp(-(dists**2) / (2 * kernel_width**2))

    # Step 4: fit weighted local Ridge
    local_m = Ridge(alpha=1.0)
    local_m.fit(X_pert, y_pert, sample_weight=weights)

    lp = float(np.ravel(local_m.predict(x.reshape(1,-1)))[0])
    gp = float(np.ravel(lo + np.clip(model.predict(x.reshape(1,-1)),0,1)*(hi-lo))[0])
    return local_m.coef_, lp, gp

# Explain 3 selected customers per store (low, mid, high error)
print("Computing LIME explanations...")
for ts in STORES:
    r = pipeline[ts]; errs = r["errors"]
    sel = [np.argsort(errs)[5],       # low-error
           np.argsort(errs)[len(errs)//2],  # mid-error
           np.argsort(errs)[-5]]      # high-error
    lime_results = []
    for idx in sel:
        coefs, lp, gp = lime_explain(r["model"], r["Xt"][idx], r["lo"], r["hi"])
        lime_results.append({"idx":idx,"coefs":coefs,"local_pred":lp,"global_pred":gp})
    pipeline[ts]["lime"] = lime_results
print("Done.")

## 11. LIME explanation plots — local vs global attribution

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 12))
fig.suptitle("LIME local explanations: low / mid / high error customers per store",
             fontweight="bold")

err_labels = ["Low-error customer", "Mid-error customer", "High-error customer"]

for col_i, (ts, slbl, col) in enumerate(zip(STORES, LABELS, COLORS)):
    r = pipeline[ts]
    for row_i, lime_r in enumerate(r["lime"]):
        ax     = axes[row_i, col_i]
        idx    = lime_r["idx"]
        actual = r["yt"][idx]
        g_pred = lime_r["global_pred"]
        l_coefs= lime_r["coefs"]

        # Global (linear) attribution at this point
        g_coefs = r["contribs"][idx]

        # Compare top features: LIME vs linear attribution
        top_by_lime = np.argsort(np.abs(l_coefs))[::-1][:8]
        x_pos = np.arange(len(top_by_lime))

        ax.bar(x_pos-0.2, g_coefs[top_by_lime]/1000, 0.35,
               label="Linear (global)", color=col, alpha=0.6, edgecolor="white")
        ax.bar(x_pos+0.2, l_coefs[top_by_lime]/1000, 0.35,
               label="LIME (local)",    color=col, alpha=1.0, edgecolor="white",
               hatch="//")
        ax.axhline(0, color="black", linewidth=0.7)
        ax.set_xticks(x_pos)
        ax.set_xticklabels([FEAT_NICE[i] for i in top_by_lime],
                            rotation=45, ha="right", fontsize=7.5)

        err = abs(actual - g_pred)/1000
        title = f"{slbl} | {err_labels[row_i]}\nActual=${actual/1000:.0f}k  Pred=${g_pred/1000:.0f}k  Err=${err:.1f}k"
        ax.set_title(title, fontsize=7.5, fontweight="bold", color=col)
        if col_i == 0: ax.set_ylabel("Contribution ($k)")
        if row_i == 0: ax.legend(fontsize=6.5, loc="upper right")

plt.tight_layout()
plt.show()

## 12. Summary — what drives bad predictions?

Bringing together all three methods to answer the core question.

In [ ]:
# Build summary: for each store, report the top 3 features
# driving errors (by attribution-error correlation)
print("="*72)
print("FEATURE ATTRIBUTION SUMMARY")
print("="*72)

for ts, lbl, col in zip(STORES, LABELS, COLORS):
    r = pipeline[ts]
    contribs = r["contribs"]
    signed_errors = r["preds"] - r["yt"]
    errs = r["errors"]

    # Method: corr between attribution and signed error
    corrs = [float(spearmanr(contribs[:,j], signed_errors)[0])
             for j in range(len(FEAT_COLS))]
    top_corr = np.argsort(np.abs(corrs))[-3:][::-1]

    # Permutation importance top features
    perm_top = np.argsort(r["perm_imp_mean"])[-3:][::-1]

    print(f"\n  {lbl}")
    print(f"  MAE = ${errs.mean():,.0f}  |  "
          f"Range est. ${r['lo']:,}-${r['hi']:,}")
    print(f"  Permutation importance top 3:")
    for i in perm_top:
        print(f"    {FEAT_NICE[i]:<20}  +${r['perm_imp_mean'][i]/1000:.1f}k MAE increase when shuffled")
    print(f"  Features most correlated with over-prediction:")
    for i in top_corr:
        if corrs[i] > 0.1:
            print(f"    {FEAT_NICE[i]:<20}  r={corrs[i]:+.3f} (higher attribution -> over-predicts)")
        elif corrs[i] < -0.1:
            print(f"    {FEAT_NICE[i]:<20}  r={corrs[i]:+.3f} (lower attribution -> under-predicts)")

print("\n" + "="*72)
print("KEY INSIGHT: bad predictions are not random.")
print("They cluster around customers where the income-behavior relationship")
print("differs from what was learned in the source domain.")
print("The features with highest attribution-error correlation are the")
print("features where the source learned the wrong direction or magnitude.")